In [1]:
from bm25 import BM25Retriever
from vdb import FaissRetriever
from rrf import rrf_fusion
from langchain_ollama import ChatOllama

d:\Miniconda\envs\agent\lib\site-packages\jieba\_compat.py:18: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [13]:
from typing import Annotated, TypedDict, List, Union, Optional
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage
from operator import add
from langgraph.graph import StateGraph, END, START

# 和file一样，包含被append的信息和被write的信息
class ReActState(TypedDict):
    documents: List[dict]
    is_valid: bool
    messages: Annotated[List[BaseMessage], add]
    loop_count: int = 0

# 先尝试空转
def recall_node(state: ReActState) -> ReActState:
    user_query = state['messages'][-1].content
    # faiss_retriever = FaissRetriever(chunks_file='first10_chunks.json')
    # bm25_retriever = BM25Retriever(chunk_file='first10_chunks.json')
    # faiss_results = faiss_retriever.retrieve(user_query, top_k=60)
    # bm25_results = bm25_retriever.retrieve(user_query, top_k=60)
    # fused_results = rrf_fusion(faiss_results, bm25_results, k=60, top_k=5)
    # 还要增加rerank
    # if fused_results:
    #     return {
    #         "documents": fused_results,
    #         "messages": state['messages']
    #     }
    mock_docs = [{"chpt_id": 0, "chunk_id": 0, "chunk_text": "这是一个空的回忆节点。"}]
    show_str = '到达recall节点，未检索到相关内容，返回空内容。\n'
    print(show_str)
    return {
        "documents": mock_docs,
        "loop_count": state['loop_count'] + 1,
        "messages": [AIMessage(content=show_str)],
    }

def generate_node(state: ReActState) -> ReActState:
    # llm = ChatOllama(
    #     model='hoangquan456/qwen3-nothink:8b',
    #     temperature=0,
    # )
    # response = llm.invoke(state['messages'])
    response = "这是一个空的生成节点回应。"
    show_str = '到达generate节点，生成回应内容。\n'
    print(show_str)
    return {
        'messages': [AIMessage(content=response)],
    }
def feedback_node(state: ReActState) -> ReActState:
    response = "这是一个空的反馈节点回应。"
    show_str = '到达feedback节点，进行反馈处理。\n'
    print(show_str)
    return {
        'messages': [AIMessage(content=response)],
    }
def router(state: ReActState):
    if state['loop_count'] >= 3:
        return END
    else:
        return "recall"

In [14]:
workflow = StateGraph(ReActState)

workflow.add_node("recall", recall_node)
workflow.add_node("generate", generate_node)
workflow.add_node("feedback", feedback_node)

workflow.set_entry_point("recall")
workflow.add_edge("recall", "generate")
workflow.add_edge("generate", "feedback")
workflow.add_conditional_edges("feedback", router)

app = workflow.compile()
initial_state: ReActState = {
    "documents": [],
    "is_valid": True,
    "messages": [HumanMessage(content="请根据回忆的内容，回答我的问题：地球是圆的吗？")],
    "loop_count": 0,
}
final_state = app.invoke(initial_state)
print("Final messages:")
for msg in final_state['messages']:
    print(msg)

到达recall节点，未检索到相关内容，返回空内容。

到达generate节点，生成回应内容。

到达feedback节点，进行反馈处理。

到达recall节点，未检索到相关内容，返回空内容。

到达generate节点，生成回应内容。

到达feedback节点，进行反馈处理。

到达recall节点，未检索到相关内容，返回空内容。

到达generate节点，生成回应内容。

到达feedback节点，进行反馈处理。

Final messages:
content='请根据回忆的内容，回答我的问题：地球是圆的吗？' additional_kwargs={} response_metadata={}
content='到达recall节点，未检索到相关内容，返回空内容。\n' additional_kwargs={} response_metadata={}
content='这是一个空的生成节点回应。' additional_kwargs={} response_metadata={}
content='这是一个空的反馈节点回应。' additional_kwargs={} response_metadata={}
content='到达recall节点，未检索到相关内容，返回空内容。\n' additional_kwargs={} response_metadata={}
content='这是一个空的生成节点回应。' additional_kwargs={} response_metadata={}
content='这是一个空的反馈节点回应。' additional_kwargs={} response_metadata={}
content='到达recall节点，未检索到相关内容，返回空内容。\n' additional_kwargs={} response_metadata={}
content='这是一个空的生成节点回应。' additional_kwargs={} response_metadata={}
content='这是一个空的反馈节点回应。' additional_kwargs={} response_metadata={}
